# scib-validity quickstart

CKA null saturation bounds and source classifier confidence (SCC) for single-cell embedding evaluation.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/elliottower/scib-construct-validity/blob/main/notebooks/quickstart.ipynb)

In [ ]:
!pip install -q scib-validity

## CKA null saturation

CKA on cell-type centroids has a closed-form null expectation:

$$\mathbb{E}[\text{CKA}] \approx 1 - 1.06 \cdot \frac{k-1}{d+k}$$

At foundation-model dimensionality ($d \geq 512$), this floor exceeds every trained-model CKA score.

In [ ]:
from scib_validity import cka_null, cka_certifiable
import numpy as np

k = 15  # cell types
dims = [10, 50, 100, 256, 512, 1024]

print(f"{'d':>6}  {'E[CKA] null':>12}  {'CKA=0.95 certifiable?':>22}")
print("-" * 46)
for d in dims:
    null = cka_null(k, d)
    cert = cka_certifiable(0.95, k, d)
    print(f"{d:>6}  {null:>12.4f}  {str(cert):>22}")

At $d=50$, CKA = 0.95 exceeds the null floor. At $d=512$, the null floor itself is above 0.97, so CKA = 0.95 is *below random*.

## Source classifier confidence (SCC)

SCC trains a cell-type classifier on source embeddings and reports the mean maximum predicted probability on target cells. It directly measures whether learned decision boundaries transfer.

In [ ]:
from scib_validity import scc, scc_multi
from sklearn.datasets import make_classification

# Simulate source and target embeddings with shared cell types
rng = np.random.default_rng(42)
n_types = 10
d = 512
n_per_type = 200

# Source: well-separated clusters
centers_source = rng.standard_normal((n_types, d)) * 3
X_source = np.vstack([centers_source[i] + rng.standard_normal((n_per_type, d)) * 0.5
                       for i in range(n_types)])
labels_source = np.repeat(np.arange(n_types), n_per_type)

# Target (good transfer): same structure with small shift
X_target_good = np.vstack([centers_source[i] + rng.standard_normal((n_per_type, d)) * 0.8
                            for i in range(n_types)])

# Target (bad transfer): scrambled structure
X_target_bad = rng.standard_normal((n_types * n_per_type, d))

labels_target = np.repeat(np.arange(n_types), n_per_type)

score_good = scc(X_source, X_target_good, labels_source, labels_target)
score_bad = scc(X_source, X_target_bad, labels_source, labels_target)

print(f"SCC (good transfer): {score_good:.3f}")
print(f"SCC (bad transfer):  {score_bad:.3f}")

## Cross-classifier stress test

Run SCC with all four classifier families to rule out shared-machinery confounding.

In [ ]:
scores = scc_multi(X_source, X_target_good, labels_source, labels_target)

print(f"{'Classifier':>12}  {'SCC':>6}")
print("-" * 22)
for name, score in scores.items():
    print(f"{name:>12}  {score:>6.3f}")

## Empirical CKA null distribution

Verify the closed-form bound against Monte Carlo simulation.

In [ ]:
from scib_validity.metrics.cka_null import cka_null_empirical

k, d = 15, 512
empirical = cka_null_empirical(k, d, n_trials=5000)
analytic = cka_null(k, d)

print(f"Analytic E[CKA]:  {analytic:.4f}")
print(f"Empirical mean:   {empirical['mean']:.4f} +/- {empirical['std']:.4f}")
print(f"Empirical 95% CI: [{empirical['ci_lower']:.4f}, {empirical['ci_upper']:.4f}]")